# Accessing Data from a Physics 180E HDF5 File

In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from bapsflib import phys180E

In [ ]:
plt.rcParams.update(
    {"figure.figsize": [10, 0.6 * 10], "font.size": 16}
)

Let's define the file name and path to the HDF5 file.

In [ ]:
# _HERE is the location of this jupyter notebook
_HERE = Path.cwd()
_FILENAME = "sample_phys180e_axial_bdot.hdf5"
_FILE = (_HERE / _FILENAME).resolve()

print(
    f"File name & path: {_FILE}\n"
    f"File exists: {_FILE.exists()}"
)

In [ ]:
all_items = [str(p) for p in _HERE.iterdir()]
print("\n".join(all_items))

The HDF5 file can now be opened using the `File` class on `phys180e`.

In [ ]:
f = phys180E.File(_FILE)

`f` is now an object instance of the `phys180e.File` class.  All the functionality provided by the `File` class is accessible via dot access.

For example, a summary of the digitizer can shown by...

In [ ]:
f.digitizers

Here, the digitizer used to recorded data was the `'LeCroy_scope'` and all 4 channels recorded data with a record length of `2500`.

And a more details digitizer report can be accessed on the `overview` attribute.

In [ ]:
f.overview.report_digitizers()

## Reading Digitizer Data

Digitizer data can be retrieved by using the `read_data()` method.

For example, lets retrieve all the recorded data for channel 2 of the digitizer.

In [ ]:
data = f.read_data(board=0, channel=2, silent=True)
data.dtype

The retrieved data has three fields:

- `"shotnum"`:  This is the shot number array.
- `"signal"`:  This is the recorded signal data.
- `"xyz"`: This field contains probe positions.  However, we have not read out any position data, so this will be covered later in the notebook.

The total number of shots for this HDF5 file is...

In [ ]:
data["shotnum"].size

Let's say we wanted to plot the signal for shot number 49. We could do something like...

In [ ]:
mask = data["shotnum"] == 49
data49 = data["signal"][mask, ...]

The ellipses `...` is requires since `data["shotnum"]` (and thus `mask`) have a shape of `(50,)`, where as `data["signal"]` has a shape of `(50, 2500)`.

In [ ]:
plt.plot(data49.squeeze());

A single record can also be read directly by `read_data()`.

In [ ]:
data35 = f.read_data(0, 2, shotnum=35, silent=True)

plt.plot(data35["signal"].squeeze());

## Reading the Time Series

The `File` object also provides a convienece method, `get_time_array()`, to retrieve the time array associated with the digitized data.  This method takes just one argument, the data array.

In [ ]:
# let's just work with data35 from now on
data = data35

# get the time array
time = f.get_time_array(data)

plt.plot(time, data["signal"].squeeze())
ax = plt.gca()
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Voltge (V)");

The time array can also be retrieved before any data is extracted.

In [ ]:
digitized_info = f.get_digitizer_specs(0, 2, silent=True)
time_from_digi_info = f.get_time_array(digitized_info)

# see... both arrays are equal
np.allclose(time, time_from_digi_info)

## Reading Position Information

The digitized data is often associated with a probe that is connected to an automated probe drive (or a control device).  This probe drive also records this position data in the HDF5 file.  Similarly to `f.digitizers`, `f.controls` will yield a brief summary of the mapped control devices...

In [ ]:
f.controls

Here the control device is call `"180E_positions"` and will map the the `"xyz"` field briefly mentioned when discussing the extracted digitizer data.

A more detailed report can also be displayed using the `overview` attribute.

In [ ]:
f.overview.report_controls()

To retrieving the control data can be done using the `read_control()` method.

In [ ]:
cdata = f.read_controls(["180E_positions"], silent=True)
cdata.dtype

The retrieved control data has three fields:

- `"shotnum"`:  This is the shot number array.
- `"xyz"`: This field contains probe positions.

So, the position for shot number 35 is...

In [ ]:
print(
    f"Shot number: {cdata['shotnum'][34]}\n"
    f"Position [x, y, z]: {cdata['xyz'][34]}"
)

Now, this is not so convenient since the position data is not paired with the signal data.  So, `read_data` provides a mechanism for mating the digitizer and position data at the same time.

In [ ]:
all_data = f.read_data(0, 2, add_controls=["180E_positions"], shotnum=35, silent=True)

All the extracted data in `all_data` is the same as what we have extracted before.

In [ ]:
print(
    f"Shot number is the same: {np.allclose(data['shotnum'], all_data['shotnum'])}\n"
    f"Signal data is the same: {np.allclose(data['signal'], all_data['signal'])}\n"
    f"Position data is the same: {np.allclose(cdata['xyz'][34], all_data['xyz'])}\n"
)

## Close the `File` instance

The `File` instance will automatically close itself when Python garbage collects at the close of this notebook.  However, it is good practice to explicitly close the file.

In [ ]:
f.close()